# SKRIPSI: Train YOLOv12

---

Notebook ini dicopy dari [`How to Train YOLO11 Object Detection on a Custom Dataset`](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolo11-object-detection-on-custom-dataset.ipynb) dari Github Roboflow.  
  

*   Notebook ini dibuat untuk membangun model pendeteksi tanda vital di monitor fisiologis/bedside monitor.
*   Menggunakan pre-trained model **`YOLOv12`**
*   Dataset: [Dataset Skripsi_V1](https://app.roboflow.com/skripsian-bwueb/skripsi_v1/2/)

**NOTE: PASTIKAN ANDA MEMILIKI AKUN ROBOFLOW UNTUK MENGAKSES DAN MENGGUNAKAN DATASET**

## **STEP 1: Environment and Library Setup**

### 1.1 - Configure API keys
- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Kaggle, go to the `Add-ons` and click on `Secrets` (🔑). Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

### 1.2 - Configure GPU

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Settings` -> `Accelerator`, set it to `GPU P100`, and then click `Save`.

In [1]:
!nvidia-smi

Thu Apr 16 16:17:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 1.3 - Create a `HOME` constant.

In [2]:
import os
HOME = os.getcwd()
print(HOME)

/kaggle/working


### 1.4 - Install Dependencies

In [3]:
!git clone https://github.com/sunsmarterjie/yolov12

Cloning into 'yolov12'...
remote: Enumerating objects: 1169, done.
remote: Total 1169 (delta 0), reused 0 (delta 0), pack-reused 1169 (from 1)
Receiving objects: 100% (1169/1169), 1.95 MiB | 6.72 MiB/s, done.
Resolving deltas: 100% (531/531), done.


In [4]:
import os
os.chdir('yolov12')

!pip install -r requirements.txt
!pip install -e .

ERROR: flash_attn-2.7.3+cu11torch2.2cxx11abiFALSE-cp311-cp311-linux_x86_64.whl is not a supported wheel on this platform.
Obtaining file:///kaggle/working/yolov12
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.63-0.editable-py3-none-any.whl size=20193 sha256=912a31db8694987fd868745d37ca4e2739ad0499a236a9c8381f5b202dc496e3
  Stored in directory: /tmp/pip-ephem-wheel-cache-3h7fu2kh/wheels/e1/69/6a/b9decd48b276a3db8e45bdfecfc351a6c467173183690b1ee8
Successfully built ultralytics


In [5]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (4 CPUs, 31.4 GB RAM, 6841.9/8062.4 GB disk)


In [6]:
!pip install wandb supervision roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 45.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 22.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 106.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the

In [7]:
!yolo settings wandb=true

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
JSONDict("/root/.config/yolov12/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/kaggle/working/datasets",
  "weights_dir": "/kaggle/working/yolov12/weights",
  "runs_dir": "/kaggle/working/yolov12/runs",
  "uuid": "1bfc3e992d24318da58ddee183be5bf9388a31f26bab1738e986ec4d297417ff",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": true,
  "wandb": true,
  "vscode_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


## **STEP 2: Download Dataset via Code from Roboflow**

### 2.1 - Get datasets from Roboflow

In [8]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
ROBOFLOW_API_KEY = user_secrets.get_secret("ROBOFLOW_API_KEY")

os.chdir("/kaggle/working/")

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("skripsian-bwueb").project("skripsi_v1")
version = project.version(2)
dataset = version.download("yolov12")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Skripsi_V1-2 in yolov12:: 100%|██████████| 3135/3135 [00:00<00:00, 8780.71it/s]


In [9]:
os.chdir("/kaggle/working/Skripsi_V1-2")
DATASET_PATH = os.getcwd()
print(DATASET_PATH)

/kaggle/working/Skripsi_V1-2


## **STEP 3: Start Custom Training!**

### 3.1 - Train with Code

In [10]:
# Setup wandb for monitoring
import wandb
from kaggle_secrets import UserSecretsClient

# Mengambil API Key dari Kaggle Secrets secara aman
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

# Login ke W&B
wandb.login(key=wandb_api)

# Inisialisasi proyek di W&B
wandb.init(project="skripsi-monitor-medis", name="yolov12n_baseline")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aizarhafizh (aizarhafizh-universitas-gadjah-mada-library) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run f0zts8db
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/Skripsi_V1-2/wandb/run-20260416_161836-f0zts8db
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run yolov12n_baseline
wandb: ⭐️ View project at https://wandb.ai/aizarhafizh-universitas-gadjah-mada-library/skripsi-monitor-medis
wandb: 🚀 View run at https://wandb.ai/aizarhafizh-univ

In [11]:
os.chdir(HOME)

# Pastikan Anda sudah menginstall versi terbaru: pip install ultralytics
from ultralytics import YOLO

# 1. Inisialisasi Model
# Gunakan arsitektur Nano untuk eksperimen awal yang ringan
model = YOLO('yolo12n.pt') 

# 2. Path ke file konfigurasi dataset
# Jika di Kaggle, sesuaikan path ini dengan folder hasil download Roboflow Anda
dataset_yaml = os.path.join(DATASET_PATH, "data.yaml")

# 3. Eksekusi Pelatihan dengan Konfigurasi Hyperparameter Lengkap
results = model.train(
    # ==========================================
    # A. PARAMETER DASAR (BASIC CONFIGURATION)
    # ==========================================
    data=dataset_yaml,
    epochs=100,             # Total iterasi pelatihan (100 sudah cukup untuk 1500+ data)
    imgsz=640,              # Ukuran resolusi gambar input (standar YOLO)
    batch=16,               # Jumlah gambar per iterasi (naikkan ke 32 jika memori GPU besar)
    device=[0,1],               # Gunakan GPU 0 (Kaggle). Jika pakai 2 GPU T4, ketik list: [0,1]
    workers=8,              # Jumlah thread CPU untuk memuat data (8 aman untuk Kaggle)
    project='exp_yolov12n',   # Direktori utama penyimpanan hasil
    name='baseline', # Nama folder eksperimen saat ini
    
    # ==========================================
    # B. PARAMETER OPTIMASI & LEARNING (TUNABLE)
    # ==========================================
    # optimizer='auto',       # Pilihan: 'SGD', 'Adam', 'AdamW'. 'auto' sangat disarankan.
    # lr0=0.01,               # Initial Learning Rate (Standar AdamW: 0.001, SGD: 0.01)
    # lrf=0.01,               # Final Learning Rate (fraksi dari lr0, misal: 0.01 * lr0)
    # # momentum=0.937,         # Momentum gradien (mencegah training tersendat di lokal minima)
    # # weight_decay=0.0005,    # Penalti kompleksitas model (mencegah overfitting)
    # # warmup_epochs=3.0,      # Jumlah epoch untuk "pemanasan" lr dari rendah ke normal
    # # warmup_momentum=0.8,    # Momentum awal saat fase pemanasan
    # patience=10,            # Early Stopping: berhenti jika metrik tidak membaik selama X epoch
    
    # ==========================================
    # C. PARAMETER AUGMENTASI (SANGAT KRITIS UNTUK TEKS)
    # ==========================================
    # 1. Augmentasi Haram (WAJIB 0.0 karena objek adalah angka/teks)
#     fliplr=0.0,             # JANGAN DIBALIK HORIZONTAL (73 -> E7)
#     flipud=0.0,             # JANGAN DIBALIK VERTIKAL (6 -> 9)
#     mixup=0.0,              # JANGAN DITUMPUK (Angka berbayang merusak bounding box)
#     copy_paste=0.0,         # Tidak relevan untuk data teks utuh
    
#     # 2. Augmentasi Esensial (Bisa Anda tuning nilainya)
#     hsv_h=0.015,            # Variasi Warna (Hue). Kecil saja karena warna monitor medis baku
#     hsv_s=0.7,              # Variasi Saturasi. (Membantu model tahan terhadap layar pudar)
#     hsv_v=0.4,              # Variasi Kecerahan (Value). (Membantu model tahan cahaya silau/gelap)
#     degrees=5.0,            # Rotasi maksimal gambar (derajat). 5 sudah cukup untuk kemiringan foto.
#     translate=0.1,          # Geser gambar secara vertikal/horizontal (10%).
#     scale=0.5,              # Zoom in/out gambar (+- 50%). Membantu mendeteksi monitor dari jauh.
#     shear=2.0,              # Tarikan distorsi trapesium (2 derajat).
#     perspective=0.001,      # Distorsi 3D ringan (mensimulasikan foto dari sudut miring).
#     mosaic=0.0,             # Menggabungkan 4 gambar. (Standar 1.0, ubah ke 0.0 jika mAP memburuk).
#     erasing=0.0,            # Random Erasing (menghapus kotak kecil acak). Simulasikan layar tertutup kabel/tangan.

#     # ==========================================
#     # D. PARAMETER SISTEM & REPRODUCIBILITY
#     # ==========================================
#     save=True,              # Simpan bobot (weights) model terbaik dan terakhir
#     save_period=-1,         # Simpan checkpoint setiap X epoch (-1 artinya hanya best dan last)
#     cache=False,            # Simpan data di RAM ('ram' atau 'disk'). False saja jika RAM Kaggle terbatas.
#     deterministic=True,     # Memaksa hasil training bisa diulang (reproducible) dengan seed yang sama
#     seed=42                 # Angka seed untuk mengunci nilai acak awal
)

# 4. Validasi Model Terbaik
print("Pelatihan selesai. Memulai validasi...")
metrics = model.val()
print(f"mAP50: {metrics.box.map50}")

100%|██████████| 5.34M/5.34M [00:00<00:00, 23.9MB/s]


New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                        CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolo12n.pt, data=/kaggle/working/Skripsi_V1-2/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=[0, 1], workers=8, project=exp_yolov12n, name=baseline, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=F

100%|██████████| 755k/755k [00:00<00:00, 5.47MB/s]
E0000 00:00:1776356336.372843      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776356336.419377      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776356336.781849      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776356336.781883      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776356336.781886      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776356336.781888      23 computation_placer.cc:177] comput

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  2    174720  ultralytics.nn.modules.block.A2C2f           [128, 128, 2, True, 4]        
  7                  -1  1    295424  ultralytics

E0000 00:00:1776356362.431400     129 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776356362.439151     129 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776356362.459194     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776356362.459218     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776356362.459221     129 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776356362.459224     129 computation_placer.cc:177] computation placer already registered. Please check linka

...

                   all        235       1206      0.997      0.996      0.995      0.952
                BP_DIA        191        191      0.996          1      0.995      0.945
               BP_MEAN        192        193      0.995      0.986      0.994       0.91
                BP_SYS        191        191          1      0.995      0.995      0.946
                    HR        222        222      0.996          1      0.995       0.98
                    RR        199        199      0.999          1      0.995      0.959
                  SPO2        209        210      0.997      0.995      0.995      0.971
Speed: 0.2ms preprocess, 4.5ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to exp_yolov12n/baseline


wandb: uploading history steps 96-97, summary, console lines 517-523; uploading artifact run_cb7gbeer_model; updating run metadata; uploading artifact run-cb7gbeer-curvesPrecision-RecallB_table-KvZCjg; uploading artifact run-cb7gbeer-curvesF1-ConfidenceB_table-XEpl9g (+ 2 more)
wandb: uploading history steps 96-97, summary, console lines 517-523; uploading artifact run_cb7gbeer_model; uploading artifact run-cb7gbeer-curvesPrecision-RecallB_table-KvZCjg; uploading artifact run-cb7gbeer-curvesF1-ConfidenceB_table-XEpl9g; uploading artifact run-cb7gbeer-curvesPrecision-ConfidenceB_table-FTV6Vw (+ 1 more)
wandb: uploading history steps 96-97, summary, console lines 517-523; uploading artifact run_cb7gbeer_model; uploading artifact run-cb7gbeer-curvesPrecision-RecallB_table-KvZCjg; uploading artifact run-cb7gbeer-curvesF1-ConfidenceB_table-XEpl9g; uploading artifact run-cb7gbeer-curvesPrecision-ConfidenceB_table-FTV6Vw (+ 21 more)
wandb: uploading history steps 96-97, summary, console lines

Pelatihan selesai. Memulai validasi...
Ultralytics 8.3.63 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                        CUDA:1 (Tesla T4, 14913MiB)
YOLOv12n summary (fused): 376 layers, 2,539,466 parameters, 0 gradients, 6.3 GFLOPs


val: Scanning /kaggle/working/Skripsi_V1-2/valid/labels.cache... 235 images, 0 backgrounds, 0 corrupt: 100%|██████████| 235/235 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:03<00:00,  3.98it/s]


                   all        235       1206      0.997      0.996      0.995      0.953
                BP_DIA        191        191      0.996          1      0.995      0.948
               BP_MEAN        192        193      0.995      0.986      0.994      0.913
                BP_SYS        191        191          1      0.995      0.995      0.944
                    HR        222        222      0.996          1      0.995       0.98
                    RR        199        199      0.999          1      0.995       0.96
                  SPO2        209        210      0.997      0.995      0.995      0.975
Speed: 0.3ms preprocess, 9.0ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to exp_yolov12n/baseline2
mAP50: 0.9948878613945579


### 3.2 - Save train results

In [12]:
# !zip -r yolov12_train_deafult_augment.zip /kaggle/working/runs/

### 3.3 - Examining train results  
**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [13]:
# !ls {HOME}/runs/detect/train/
# train_folder = f'{HOME}/runs/detect/train' # Change this folder according to your recent train folder name
# weight_folder = f'{train_folder}/weights'

In [14]:
# # Confusion matrix
# from IPython.display import Image as IPyImage

# IPyImage(filename=f'{train_folder}/confusion_matrix.png', width=600)

In [15]:
# # Train graph results
# IPyImage(filename=f'{train_folder}/results.png', width=600)

In [16]:
# Validation image batch results
# IPyImage(filename=f'{train_folder}/val_batch0_pred.jpg', width=600)

## Validate fine-tuned model

In [17]:
# !yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

## Inference with custom model

In [18]:
# !yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

**NOTE:** Let's take a look at few results.

In [19]:
# import glob
# import os
# from IPython.display import Image as IPyImage, display

# latest_folder = max(glob.glob(f'{HOME}/runs/detect/predict*/'), key=os.path.getmtime)
# for img in glob.glob(f'{latest_folder}/*.jpg'):
#     display(IPyImage(filename=img, width=600))
#     print("\n")